# TrashScan — Path A simplificado - YOLO v11m sem NMS

Este notebook assume que a estrutura **já está baixada/criada no volume** do RunPod:

- `/workspace/.venv`
- `/workspace/TrashScan`
- `/workspace/TACO`
- `/workspace/external_datasets`
- `/workspace/processed_4cls`
- `/workspace/runs`
- `/workspace/results_path_A`

O foco aqui é rodar o fluxo do **Path A** usando os scripts `.py` do projeto, sem refazer downloads e sem refazer merge/preprocessamento por padrão.

> Observação: os pesos dos modelos **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e devem estar disponíveis em `runs/path_A`.


## 0) Antes de abrir/rodar o notebook

No terminal do pod, ative a venv do volume:

```bash
cd /workspace
source .venv/bin/activate
```

Se precisar reinstalar dependências:

```bash
cd /workspace/TrashScan/env
python -m pip install -r environment.txt
```

No VSCode/Jupyter, selecione o kernel correspondente a:

```bash
/workspace/.venv/bin/python
```

Para confirmar dentro do notebook, rode a célula abaixo e confira `sys.executable`.


## 1) Imports, paths e utilitários

In [1]:
from pathlib import Path
import os
import sys
import shlex
import subprocess
import shutil

# Caminhos principais no RunPod
WORKSPACE = Path('/workspace')
REPO_ROOT = WORKSPACE / 'TrashScan'

DATA_DIR = REPO_ROOT / 'data'
TRAIN_DIR = REPO_ROOT / 'train' / 'paths'
EVAL_DIR = REPO_ROOT / 'eval'

EXTERNAL_DIR = WORKSPACE / 'external_datasets'
TACO_DIR = WORKSPACE / 'TACO'
PROCESSED_DIR = WORKSPACE / 'processed_5cls'
DATASET_YAML_PATH_A = PROCESSED_DIR / 'dataset_path_A.yaml'

# Treino fica no disco local do pod para evitar I/O lento e cache pesado no volume.
# Se o pod for deletado, salve o essencial de volta no volume antes.
RUNS_PATH_A_DIR = WORKSPACE / 'runs' / 'path_A_5cls'
MLFLOW_DIR = Path('/root/mlflow')

# Resultados finais persistentes no volume
RESULTS_PATH_A_DIR = WORKSPACE / 'results_path_A_no_NMS_5'

for p in [RUNS_PATH_A_DIR, MLFLOW_DIR, RESULTS_PATH_A_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Python             =', sys.executable)
print('REPO_ROOT          =', REPO_ROOT)
print('TACO_DIR           =', TACO_DIR)
print('EXTERNAL_DIR       =', EXTERNAL_DIR)
print('PROCESSED_DIR      =', PROCESSED_DIR)
print('DATASET_YAML_PATH_A=', DATASET_YAML_PATH_A)
print('RUNS_PATH_A_DIR    =', RUNS_PATH_A_DIR)
print('MLFLOW_DIR         =', MLFLOW_DIR)
print('RESULTS_PATH_A_DIR =', RESULTS_PATH_A_DIR)


def run_cmd(cmd, cwd=WORKSPACE, env=None):
    """Roda comandos de forma previsível no notebook."""
    if isinstance(cmd, str):
        print('$', cmd)
        return subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)

    print('$', ' '.join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)


Python             = /workspace/.venv/bin/python
REPO_ROOT          = /workspace/TrashScan
TACO_DIR           = /workspace/TACO
EXTERNAL_DIR       = /workspace/external_datasets
PROCESSED_DIR      = /workspace/processed_5cls
DATASET_YAML_PATH_A= /workspace/processed_5cls/dataset_path_A.yaml
RUNS_PATH_A_DIR    = /workspace/runs/path_A_5cls
MLFLOW_DIR         = /root/mlflow
RESULTS_PATH_A_DIR = /workspace/results_path_A_no_NMS_5


## 2) Verificação dos arquivos principais

In [2]:
required_paths = [
    DATA_DIR / 'merge_datasets.py',
    DATA_DIR / 'preprocess.py',
    TRAIN_DIR / 'train_path_A_no_NMS.py',
    EVAL_DIR / 'evaluate.py',
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print('Arquivos ausentes:')
    for m in missing:
        print(' -', m)
    raise FileNotFoundError('Há scripts ausentes no clone. Veja a lista acima.')

print('Todos os scripts principais foram encontrados.')
print('Dataset YAML existe?', DATASET_YAML_PATH_A.exists())


Todos os scripts principais foram encontrados.
Dataset YAML existe? True


## 3) Configuração de GPU e treino

In [3]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = '0'
else:
    gpu_name = 'cpu'
    DEVICE = 'cpu'

# Ajuste se necessário
EPOCHS = 300
BATCH = 16
IMGSZ = 640
PATIENCE = 15

# Pesos já calculados anteriormente: yolov11m, yolov8m, yolov9s.
# Se rodar a célula de treino abaixo, esses modelos serão treinados novamente.
MODELS = [
    'yolov11m_o2o'
]

print('Dispositivo:', gpu_name)
print('DEVICE     :', DEVICE)
print('BATCH      :', BATCH)
print('EPOCHS     :', EPOCHS)
print('IMGSZ      :', IMGSZ)
print('PATIENCE   :', PATIENCE)
print('MODELS     :', MODELS)


Dispositivo: NVIDIA RTX A4500
DEVICE     : 0
BATCH      : 16
EPOCHS     : 300
IMGSZ      : 640
PATIENCE   : 15
MODELS     : ['yolov11m_o2o']


## 4) Merge dos datasets — normalmente NÃO rodar

A estrutura já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você mudar os datasets de origem ou precisar reconstruir `merged_data`.


In [ ]:
# merge_script = DATA_DIR / 'merge_datasets.py'
#
# run_cmd([
#     sys.executable, str(merge_script),
#     '--taco_root', str(TACO_DIR),
#     '--external_root', str(EXTERNAL_DIR),
#     '--output_root', str(PROCESSED_DIR),
#     '--skip_preprocess',
# ])


## 5) Pré-processamento Path A — normalmente NÃO rodar

A estrutura `train/val/test` já existe em `/workspace/processed_4cls`, então esta célula fica comentada por padrão.

Rode apenas se você apagou/recriou o dataset processado ou mudou o mapeamento de classes.


In [ ]:
# preprocess_script = DATA_DIR / 'preprocess.py'
#
# run_cmd([
#     sys.executable, str(preprocess_script),
#     '--taco_root', str(PROCESSED_DIR / 'merged_data'),
#     '--output_root', str(PROCESSED_DIR),
#     '--path', 'A',
# ])


## 6) Conferência do dataset Path A

Confirme que o YAML está apontando para o dataset correto e que usa as 4 classes esperadas.


In [4]:
import yaml

print('DATASET_YAML_PATH_A =', DATASET_YAML_PATH_A)
print('Existe?', DATASET_YAML_PATH_A.exists())

if DATASET_YAML_PATH_A.exists():
    with open(DATASET_YAML_PATH_A, 'r') as f:
        data_yaml = yaml.safe_load(f)
    print(data_yaml)


DATASET_YAML_PATH_A = /workspace/processed_5cls/dataset_path_A.yaml
Existe? True
{'path': '/workspace/processed_5cls', 'train': 'train/path_A/images', 'val': 'val/path_A/images', 'test': 'test/path_A/images', 'nc': 5, 'names': ['plastic', 'paper', 'metal', 'glass', 'other']}


## 7) Limpeza opcional de cache `.npy` criado por `cache='disk'`

Se o volume cresceu muito depois de treinar, provavelmente foram criados arquivos `.npy` dentro de `processed_4cls`. Esta célula apenas mostra o tamanho; a deleção fica comentada.

Para evitar recriar isso, no `train_path_A.py` use `cache=False` ou `cache=True`, mas não `cache='disk'`.


In [5]:
run_cmd('du -sh /workspace/processed_5cls || true')
run_cmd("find /workspace/processed_5cls -type f -name '*.npy' | wc -l")

# Para apagar caches .npy, descomente:
# run_cmd("find /workspace/processed_4cls -type f -name '*.npy' -delete")
# run_cmd('du -sh /workspace/processed_4cls || true')


$ du -sh /workspace/processed_5cls || true
34G	/workspace/processed_5cls
$ find /workspace/processed_5cls -type f -name '*.npy' | wc -l
17935


CompletedProcess(args="find /workspace/processed_5cls -type f -name '*.npy' | wc -l", returncode=0)

## 8) Treino Path A

Esta célula roda `train_path_A.py` com saída em `/workspace/runs/path_A`.

Importante: os pesos de **`yolov11m`**, **`yolov8m`** e **`yolov9s`** já foram calculados e estão em `runs`. Rode novamente apenas se quiser retreinar.


In [5]:
train_script = TRAIN_DIR / 'train_path_A_no_NMS.py'

def find_last_checkpoint(runs_dir: Path, model_key: str) -> str | None:
    ckpt = runs_dir / model_key / 'weights' / 'last.pt'
    return str(ckpt) if ckpt.exists() else None

# Na chamada do run_cmd:
cmd = [
    sys.executable, str(train_script),
    '--data',       str(DATASET_YAML_PATH_A),
    '--output',     str(RUNS_PATH_A_DIR),
    '--models',     *MODELS,
    '--epochs',     str(EPOCHS),
    '--batch',      str(BATCH),
    '--imgsz',      str(IMGSZ),
    '--device',     str(DEVICE),
    '--patience',   str(PATIENCE),
    '--mlflow_uri', str(MLFLOW_DIR),
]

# Adiciona resume se encontrar checkpoint de algum modelo
for model_key in MODELS:
    ckpt = find_last_checkpoint(RUNS_PATH_A_DIR, model_key)
    if ckpt:
        print(f"[resume] {model_key} → {ckpt}")
        cmd += ['--resume', ckpt]
        break  # ultralytics resume é por run; treine um modelo por vez ao resumir

run_cmd(cmd)


[resume] yolov11m_o2o → /workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt
$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_A_no_NMS.py --data /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/runs/path_A_5cls --models yolov11m_o2o --epochs 300 --batch 16 --imgsz 640 --device 0 --patience 15 --mlflow_uri /root/mlflow --resume /workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt


GPU  : NVIDIA RTX A4500
VRAM : 21.2 GB
Loaded class weights: {'plastic': 0.511, 'paper': 1.098, 'metal': 1.125, 'glass': 1.777, 'other': 0.489}

────────────────────────────────────────────────────────────
  Model  : yolov11m_o2o  (yolo11m.pt)
  Data   : /workspace/processed_5cls/dataset_path_A.yaml
  Epochs : 300   Batch : 16   imgsz : 640
────────────────────────────────────────────────────────────
  Resuming from: /workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt
New https://pypi.org/project/ultralytics/8.4.51 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.2 🚀 Python-3.11.10 torch-2.3.1+cu121 CUDA:0 (NVIDIA RTX A4500, 20171MiB)
engine/trainer: task=detect, mode=train, model=/workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt, data=/workspace/processed_5cls/dataset_path_A.yaml, epochs=300, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=10, cache=disk, device=0, workers=8, project=/workspace/runs/path_A_5cls, name=yolov11m_o2o, ex

train: Scanning /workspace/processed_5cls/train/path_A/labels.cache... 18294 images, 4 backgrounds, 0 corrupt: 100%|██████████| 18294/18294 [00:00<?, ?it/s]
train: Caching images (20.9GB Disk): 100%|██████████| 18294/18294 [00:26<00:00, 691.94it/s]
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01), CLAHE(p=0.01, clip_limit=(1, 4.0), tile_grid_size=(8, 8))


val: Scanning /workspace/processed_5cls/val/path_A/labels.cache... 1392 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1392/1392 [00:00<?, ?it/s]
val: Caching images (1.6GB Disk): 100%|██████████| 1392/1392 [00:01<00:00, 761.54it/s]


...

    239/300      9.45G     0.6478     0.7613      1.088         19        640: 100%|██████████| 1144/1144 [04:21<00:00,  4.37it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:09<00:00,  4.84it/s]


                   all       1392       3446      0.782      0.668      0.713      0.491
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 224, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

27 epochs completed in 2.044 hours.
Optimizer stripped from /workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt, 40.6MB
Optimizer stripped from /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt, 40.6MB

Validating /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt...
Ultralytics 8.3.2 🚀 Python-3.11.10 torch-2.3.1+cu121 CUDA:0 (NVIDIA RTX A4500, 20171MiB)
YOLO11m summary (fused): 303 layers, 20,033,887 parameters, 0 gradients, 67.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 44/44 [00:17<00:00,  2.47it/s]


                   all       1392       3446      0.791       0.66      0.704      0.492
               plastic        767       1411      0.817      0.643      0.717      0.479
                 paper        202        266      0.754      0.729       0.69       0.49
                 metal        182        282      0.812      0.745      0.793       0.56
                 glass         54        107      0.748      0.561      0.612      0.355
                 other        622       1380      0.822      0.624      0.706      0.576
Speed: 0.1ms preprocess, 9.6ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /workspace/runs/path_A_5cls/yolov11m_o2o
MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
  Latency: 33.24 ms/image

  [yolov11m_o2o] mAP50=0.7035  mAP50-95=0.4921  latency=33.24ms

PATH A  —  YOLO BENCHMARK SUMMARY
              mAP50  mAP50_95  precision  recall  box_loss  cls_loss  latency_ms
model                          

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_A_no_NMS.py', '--data', '/workspace/processed_5cls/dataset_path_A.yaml', '--output', '/workspace/runs/path_A_5cls', '--models', 'yolov11m_o2o', '--epochs', '300', '--batch', '16', '--imgsz', '640', '--device', '0', '--patience', '15', '--mlflow_uri', '/root/mlflow', '--resume', '/workspace/runs/path_A_5cls/yolov11m_o2o/weights/last.pt'], returncode=0)

## 9) Conferir pesos disponíveis

O avaliador espera encontrar os modelos no formato:

```text
/root/runs/path_A/<modelo>/weights/best.pt
```


In [6]:
best_weights = sorted(RUNS_PATH_A_DIR.glob('*/weights/best.pt'))
print(f'Pesos encontrados em {RUNS_PATH_A_DIR}: {len(best_weights)}')
for p in best_weights:
    print(' -', p)

if not best_weights:
    raise FileNotFoundError(f'Nenhum best.pt encontrado em {RUNS_PATH_A_DIR}')


Pesos encontrados em /workspace/runs/path_A_5cls: 3
 - /workspace/runs/path_A_5cls/yolov11m/weights/best.pt
 - /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
 - /workspace/runs/path_A_5cls/yolov8m/weights/best.pt


## 10) Resumo rápido dos treinos

In [7]:
run_cmd([
    sys.executable, str(TRAIN_DIR / 'train_path_A_no_NMS.py'),
    '--summarize',
    '--output', str(RUNS_PATH_A_DIR),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/train/paths/train_path_A_no_NMS.py --summarize --output /workspace/runs/path_A_5cls

PATH A  —  YOLO BENCHMARK SUMMARY
              mAP50  mAP50_95  precision  recall  box_loss  cls_loss  latency_ms
model                                                                           
yolov11m_o2o 0.7035    0.4921     0.7907  0.6603    0.0000    0.0000     33.2420
yolov11m     0.5739    0.3858     0.6856  0.5101    0.0000    0.0000     33.4470


CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/train/paths/train_path_A_no_NMS.py', '--summarize', '--output', '/workspace/runs/path_A_5cls'], returncode=0)

## 11) Avaliação Path A

In [15]:
evaluate_script = EVAL_DIR / 'evaluate.py'

run_cmd([
    sys.executable, str(evaluate_script),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR/ "sem_tta"),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
])


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate.py --path A --runs_dir /workspace/runs/path_A_5cls --data_yaml /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/results_path_A_no_NMS_5/sem_tta --device 0 --imgsz 640



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A_5cls/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:32<00:00, 14.97it/s]



  mAP@0.5    : 0.5699
  mAP@0.5:95 : 0.3381
  Precision  : 0.5613
  Recall     : 0.4599
  F1         : 0.5015
  Latency    : 10.00 ms  (100.0 FPS)
  Params     : 20.1M
  Size       : 40.6 MB
  Saved      : /workspace/results_path_A_no_NMS_5/sem_tta/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m_o2o
  Weights   : /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:12<00:00, 19.14it/s]



  mAP@0.5    : 0.7082
  mAP@0.5:95 : 0.4740
  Precision  : 0.5828
  Recall     : 0.5592
  F1         : 0.5678
  Latency    : 10.61 ms  (94.2 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A_no_NMS_5/sem_tta/individual/A_yolov11m_o2o_yolov11m_o2o.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A_5cls/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [01:05<00:00, 21.16it/s]



  mAP@0.5    : 0.6614
  mAP@0.5:95 : 0.4383
  Precision  : 0.5564
  Recall     : 0.5360
  F1         : 0.5364
  Latency    : 8.45 ms  (118.3 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A_no_NMS_5/sem_tta/individual/A_yolov8m_yolov8m.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path    model_key       run_id  imgsz  epochs_trained  stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A yolov11m_o2o yolov11m_o2o    640             239           True 0.7082    0.4740     0.5828  0.5592 0.5678     10.6110  94.2500         20.0600
    2    A      yolov8m      yolov8m    640             212           True 0.6614    0.4383     0.5564  0.5360 0.5364      8.4520 118.3200         23.2200
    3    A     yolov11m     yolov11m    640             300          False 0.5699    0.3381     0.5613  0.4600 0.5015     10.0020  99.9800         20.0600

Per-class AP@0.5:
path    model_key  

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate.py', '--path', 'A', '--runs_dir', '/workspace/runs/path_A_5cls', '--data_yaml', '/workspace/processed_5cls/dataset_path_A.yaml', '--output', '/workspace/results_path_A_no_NMS_5/sem_tta', '--device', '0', '--imgsz', '640'], returncode=0)

## 12) Avaliação TTA + WBF — opcional

A célula abaixo fica comentada porque é mais lenta. Rode apenas se quiser avaliar com test-time augmentation e weighted boxes fusion.


In [10]:
run_cmd([
    sys.executable, str(EVAL_DIR / 'evaluate.py'),
    '--path', 'A',
    '--runs_dir', str(RUNS_PATH_A_DIR),
    '--data_yaml', str(DATASET_YAML_PATH_A),
    '--output', str(RESULTS_PATH_A_DIR),
    '--device', str(DEVICE),
    '--imgsz', str(IMGSZ),
    "--use_tta_wbf",
    "--tta_scales", "512", "640", "768",
    "--tta_flip",
    "--tta_wbf_iou", "0.55",
    "--tta_skip_box_thr", "0.001",
])


$ /workspace/.venv/bin/python /workspace/TrashScan/eval/evaluate.py --path A --runs_dir /workspace/runs/path_A_5cls --data_yaml /workspace/processed_5cls/dataset_path_A.yaml --output /workspace/results_path_A_no_NMS_5 --device 0 --imgsz 640 --use_tta_wbf --tta_scales 512 640 768 --tta_flip --tta_wbf_iou 0.55 --tta_skip_box_thr 0.001



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A_5cls/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [03:01<00:00,  7.67it/s]



  mAP@0.5    : 0.5979
  mAP@0.5:95 : 0.3469
  Precision  : 0.5571
  Recall     : 0.4401
  F1         : 0.4862
  Latency    : 9.71 ms  (103.0 FPS)
  Params     : 20.1M
  Size       : 40.6 MB
  Saved      : /workspace/results_path_A_no_NMS_5/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m_o2o
  Weights   : /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [02:36<00:00,  8.88it/s]



  mAP@0.5    : 0.7284
  mAP@0.5:95 : 0.4929
  Precision  : 0.5921
  Recall     : 0.5592
  F1         : 0.5724
  Latency    : 10.34 ms  (96.8 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A_no_NMS_5/individual/A_yolov11m_o2o_yolov11m_o2o.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A_5cls/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [02:38<00:00,  8.80it/s]



  mAP@0.5    : 0.7081
  mAP@0.5:95 : 0.4637
  Precision  : 0.6023
  Recall     : 0.5575
  F1         : 0.5740
  Latency    : 7.90 ms  (126.6 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A_no_NMS_5/individual/A_yolov8m_yolov8m.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path    model_key       run_id  imgsz  epochs_trained  stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A yolov11m_o2o yolov11m_o2o    640             239           True 0.7284    0.4929     0.5920  0.5592 0.5724     10.3360  96.7500         20.0600
    2    A      yolov8m      yolov8m    640             212           True 0.7080    0.4637     0.6023  0.5575 0.5740      7.8980 126.6100         23.2200
    3    A     yolov11m     yolov11m    640             300          False 0.5979    0.3469     0.5571  0.4401 0.4862      9.7110 102.9700         20.0600

Per-class AP@0.5:
path    model_key       run

CompletedProcess(args=['/workspace/.venv/bin/python', '/workspace/TrashScan/eval/evaluate.py', '--path', 'A', '--runs_dir', '/workspace/runs/path_A_5cls', '--data_yaml', '/workspace/processed_5cls/dataset_path_A.yaml', '--output', '/workspace/results_path_A_no_NMS_5', '--device', '0', '--imgsz', '640', '--use_tta_wbf', '--tta_scales', '512', '640', '768', '--tta_flip', '--tta_wbf_iou', '0.55', '--tta_skip_box_thr', '0.001'], returncode=0)

## 13) Grid Search TTA + WBF

In [12]:
import itertools
import subprocess
import sys

# Defina os ranges (ajuste conforme necessário)
tta_wbf_iou_values = [0.45, 0.50, 0.55, 0.60, 0.65]
tta_skip_box_thr_values = [0.0001, 0.001, 0.01, 0.05]

results = []

for iou, skip_thr in itertools.product(tta_wbf_iou_values, tta_skip_box_thr_values):
    print(f"Rodando com IOU={iou}, SKIP_THR={skip_thr}")
    
    cmd = [
        sys.executable, str(EVAL_DIR / 'evaluate.py'),
        '--path', 'A',
        '--runs_dir', str(RUNS_PATH_A_DIR),
        '--data_yaml', str(DATASET_YAML_PATH_A),
        '--output', str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}"),
        '--device', str(DEVICE),
        '--imgsz', str(IMGSZ),
        "--use_tta_wbf",
        "--tta_scales", "512", "640", "768",
        "--tta_flip",
        "--tta_wbf_iou", str(iou),
        "--tta_skip_box_thr", str(skip_thr),
    ]
    
    subprocess.run(cmd)
    
    # opcional: guardar configs testadas
    results.append({
        "iou": iou,
        "skip_thr": skip_thr,
        "output_dir": str(RESULTS_PATH_A_DIR / f"iou_{iou}_skip_{skip_thr}")
    })

print("Grid search finalizado!")

Rodando com IOU=0.45, SKIP_THR=0.0001



────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m
  Weights   : /workspace/runs/path_A_5cls/yolov11m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [03:02<00:00,  7.62it/s]



  mAP@0.5    : 0.5928
  mAP@0.5:95 : 0.3424
  Precision  : 0.5530
  Recall     : 0.4132
  F1         : 0.4652
  Latency    : 10.03 ms  (99.7 FPS)
  Params     : 20.1M
  Size       : 40.6 MB
  Saved      : /workspace/results_path_A_no_NMS_5/iou_0.45_skip_0.0001/individual/A_yolov11m_yolov11m.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov11m_o2o
  Weights   : /workspace/runs/path_A_5cls/yolov11m_o2o/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [02:44<00:00,  8.49it/s]


...


  mAP@0.5    : 0.7203
  mAP@0.5:95 : 0.4870
  Precision  : 0.5931
  Recall     : 0.5606
  F1         : 0.5736
  Latency    : 9.86 ms  (101.4 FPS)
  Params     : 20.1M
  Size       : 40.5 MB
  Saved      : /workspace/results_path_A_no_NMS_5/iou_0.5_skip_0.01/individual/A_yolov11m_o2o_yolov11m_o2o.json

────────────────────────────────────────────────────────────
  Path      : A
  Model     : yolov8m
  Weights   : /workspace/runs/path_A_5cls/yolov8m/weights/best.pt
────────────────────────────────────────────────────────────


  Inference: 100%|██████████| 1392/1392 [02:31<00:00,  9.21it/s]



  mAP@0.5    : 0.7051
  mAP@0.5:95 : 0.4607
  Precision  : 0.6026
  Recall     : 0.5587
  F1         : 0.5744
  Latency    : 7.86 ms  (127.3 FPS)
  Params     : 23.2M
  Size       : 46.8 MB
  Saved      : /workspace/results_path_A_no_NMS_5/iou_0.5_skip_0.01/individual/A_yolov8m_yolov8m.json

GLOBAL BENCHMARK SUMMARY  —  all paths, ranked by mAP@0.5
 rank path    model_key       run_id  imgsz  epochs_trained  stopped_early  mAP50  mAP50_95  precision  recall     f1  latency_ms      fps  model_params_M
    1    A yolov11m_o2o yolov11m_o2o    640             239           True 0.7203    0.4870     0.5931  0.5606 0.5736      9.8630 101.3900         20.0600
    2    A      yolov8m      yolov8m    640             212           True 0.7051    0.4607     0.6026  0.5587 0.5744      7.8560 127.2900         23.2200
    3    A     yolov11m     yolov11m    640             300          False 0.5910    0.3436     0.5598  0.4423 0.4888      9.8560 101.4600         20.0600

Per-class AP@0.5:
path    m

Traceback (most recent call last):
  File "/workspace/TrashScan/eval/evaluate.py", line 47, in <module>
    import torch
  File "/workspace/.venv/lib/python3.11/site-packages/torch/__init__.py", line 1921, in <module>
    from . import _meta_registrations
  File "/workspace/.venv/lib/python3.11/site-packages/torch/_meta_registrations.py", line 9, in <module>
    from torch._decomp import (
  File "/workspace/.venv/lib/python3.11/site-packages/torch/_decomp/__init__.py", line 244, in <module>
    import torch._decomp.decompositions
  File "/workspace/.venv/lib/python3.11/site-packages/torch/_decomp/decompositions.py", line 11, in <module>
    import torch._prims as prims
  File "/workspace/.venv/lib/python3.11/site-packages/torch/_prims/__init__.py", line 3031, in <module>
    register_debug_prims()
  File "/workspace/.venv/lib/python3.11/site-packages/torch/_prims/debug_prims.py", line 40, in register_debug_prims
    @load_tensor.impl_factory()
     ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

KeyboardInterrupt: 

## 14) Checklist final

Antes de parar/deletar o pod, confirme que o que você quer manter está no volume:

- pesos essenciais em `/workspace/runs/path_A`
- resultados em `/workspace/results_path_A`
- projeto em `/workspace/TrashScan`
- dataset em `/workspace/processed_5cls`

In [16]:
run_cmd('du -sh /workspace/runs /workspace/results_path_A_no_NMS /workspace/processed_5cls /workspace/TrashScan 2>/dev/null || true')

$ du -sh /workspace/runs /workspace/results_path_A_no_NMS /workspace/processed_5cls /workspace/TrashScan 2>/dev/null || true


9.6G	/workspace/runs
36G	/workspace/processed_5cls
583M	/workspace/TrashScan


CompletedProcess(args='du -sh /workspace/runs /workspace/results_path_A_no_NMS /workspace/processed_5cls /workspace/TrashScan 2>/dev/null || true', returncode=0)